# 02 Rebuild VWAP

Load one ingested symbol/day from DuckDB, reconstruct regular VWAP from OHLCV, and plot price plus VWAP.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import plotly.graph_objects as go

from odte_flow_lab.data.duckdb_catalog import connect, register_underlying_view
from odte_flow_lab.indicators.regular_vwap import add_regular_vwap

con = connect()
register_underlying_view(con)

## Select Symbol And Day

Leave `SYMBOL` and `TRADE_DATE` as `None` to use the latest available day.

In [ ]:
SYMBOL = None
TRADE_DATE = None

available = con.sql("""
SELECT symbol, trade_date, COUNT(*) AS rows
FROM underlying_1m
GROUP BY symbol, trade_date
ORDER BY trade_date DESC, symbol
""").df()

if available.empty:
    raise ValueError("No underlying bars found. Run `python -m odte_flow_lab.data.ingest_underlying` first.")

selected = available.iloc[0]
symbol = SYMBOL or selected.symbol
trade_date = TRADE_DATE or selected.trade_date
available.head(), symbol, trade_date

In [ ]:
bars = con.execute(
    """
    SELECT *
    FROM underlying_1m
    WHERE symbol = ? AND trade_date = ?
      AND SUBSTR(bar_start_et, 12, 8) BETWEEN '09:30:00' AND '15:59:00'
    ORDER BY bar_start_utc
    """,
    [symbol, trade_date],
).df()

bars = add_regular_vwap(bars)
bars[["symbol", "trade_date", "bar_start_et", "close", "vwap", "regular_vwap"]].head()

## Price + Regular VWAP

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Candlestick(
        x=bars["bar_start_et"],
        open=bars["open"],
        high=bars["high"],
        low=bars["low"],
        close=bars["close"],
        name="Price",
    )
)
fig.add_trace(go.Scatter(x=bars["bar_start_et"], y=bars["regular_vwap"], mode="lines", name="Regular VWAP"))
fig.add_trace(go.Scatter(x=bars["bar_start_et"], y=bars["vwap"], mode="lines", name="Massive bar VWAP", opacity=0.45))
fig.update_layout(title=f"{symbol} {trade_date} Price + VWAP", xaxis_rangeslider_visible=False, height=700)
fig.show()